In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import sqlite3

# Connect to the database
conn = sqlite3.connect("./data/nutrition.db")
cur = conn.cursor()

In [2]:
search_name = widgets.Text(
    placeholder='Enter a product name',
    description='Name:',
    continuous_update=False,
)
search_brand = widgets.Text(
    placeholder='Enter a brand name',
    description='Brand:',
    continuous_update=False,
)
out        = widgets.Output()

def update(_change):
    wal_df = pd.read_sql_query("""
                       SELECT price_retail
                       FROM walmart_price
                       WHERE product_name LIKE ? AND brand LIKE ?
                       """, conn, params=(f"%{search_name.value}%", f"%{search_brand.value}%"))
    
    wf_df = pd.read_sql_query("""
                       SELECT price
                       FROM wholefoods_price
                       WHERE product_name LIKE ? AND brand LIKE ?
                       """, conn, params=(f"%{search_name.value}%", f"%{search_brand.value}%"))

    with out:
        out.clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))

        axes[0].hist(wal_df, color='#42A5F5', edgecolor='k', linewidth=0.5)
        axes[0].set_xlabel('Price ($)')
        axes[0].set_ylabel('Products')
        axes[0].set_title('Walmart Price Distribution')

        axes[1].hist(wf_df, color="#09A846", edgecolor='k', linewidth=0.5)
        axes[1].set_xlabel('Price ($)')
        axes[1].set_ylabel('Products')
        axes[1].set_title('Wholefoods Price Distribution')

        plt.tight_layout()
        plt.show()

search_name.observe(update, names='value')
search_brand.observe(update, names='value')

display(widgets.VBox([search_name, search_brand, out]))
update(None)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans

feature_selector = widgets.SelectMultiple(
    options=['nutrient_id', 'brand_owner_encoded', 'source_wf', 'source_wal', 'latitude', 'longitude'],
    value=['nutrient_id', 'brand_owner_encoded', 'source_wf', 'source_wal', 'latitude', 'longitude'],
    description='Features:',
    rows=6,
)
out        = widgets.Output()

def update(_change):
    
    encoded_nutrients = pd.read_sql_query("SELECT * FROM encoded_nutrients", conn)

    X = encoded_nutrients[list(feature_selector.value)]
    y = encoded_nutrients['price']
    
    model = RandomForestRegressor()
    model.fit(X, y)

    preds = model.predict(X)
    encoded_nutrients['residual'] = y - preds

    residuals = encoded_nutrients[['residual', 'latitude', 'longitude']].values

    kmeans = KMeans(n_clusters=30, random_state=42)
    encoded_nutrients['residual_cluster'] = kmeans.fit_predict(residuals)

    with out:
        out.clear_output(wait=True)

        plt.scatter(encoded_nutrients.nutrient_id, encoded_nutrients['residual'], c=encoded_nutrients['residual_cluster'], cmap='viridis')
        plt.xlabel('Nutrient')
        plt.ylabel('Residual')
        plt.title('Residual Clusters by Nutrient')
        plt.colorbar(label='Cluster')
        plt.show()

feature_selector.observe(update, names='value')

display(widgets.VBox([feature_selector, out]))
update(None)